## Data Acquisition: News API

In [1]:
import os
import re
import sys
import html

import pandas as pd

from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

from newsapi import NewsApiClient

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from data_processing import DataProcessing
from feature_extraction import SpacyFeatureExtraction

In [2]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

In [3]:
api_key = os.getenv('NEWS_API_KEY')

In [4]:
QUERY_CONFIGS = [
    {"query_domain": "finance", "query": "(earnings OR revenue OR inflation OR interest rates) AND (expected OR forecast OR outlook OR projected)"}, # 0
    {"query_domain": "health", "query": "(disease OR virus OR vaccine OR public health) AND (expected OR forecast OR projected OR prognosis)"},
    {"query_domain": "weather", "query": "(weather OR storm OR hurricane OR rainfall OR temperature) AND (forecast OR expected OR outlook)"},
    {"query_domain": "policy", "query": "(election OR vote OR polling OR legislation) AND (expected OR likely OR projected OR forecast)"},
    {"query_domain": "sports_nfl", "query": "NFL AND (playoffs OR draft OR season) AND (expected OR likely OR prediction OR forecast)"},
    {"query_domain": "sports_nba", "query": "NBA AND (playoffs OR finals OR season) AND (expected OR likely OR prediction OR forecast)"},
    {"query_domain": "sports_college", "query": "(NCAA OR college football OR March Madness) AND (expected OR likely OR prediction OR forecast)"}, 
    {"query_domain": "misc-tesla", "query": "Tesla OR SAE Level 4 autonomy"}, # 7

    {"query_domain": "finance", "query": "(stocks OR markets OR economy)"},
    {"query_domain": "health", "query": " (health officials OR doctors OR experts)"},
    {"query_domain": "weather", "query": "(storm OR heat OR cold OR rain)"},
    {"query_domain": "policy", "query": "(election OR court OR vote)"},
    {"query_domain": "sports_nfl", "query": "NFL"},
    {"query_domain": "sports_nba", "query": "NBA"},
    {"query_domain": "sports_college", "query": "(college football OR NCAA)"}, # 14
]

list_num = 9

# neeed 8 to 13

In [6]:
# query_string = "Tesla OR SAE Level 4 autonomy"
start_date = "2026-03-11"
end_date = "2026-04-09"
newsapi = NewsApiClient(api_key=api_key)
all_articles = newsapi.get_everything(
    q=QUERY_CONFIGS[list_num]['query'],
    language="en",
    from_param=start_date,
    to=end_date,
    sort_by="relevancy",
    page_size=100
)

In [7]:
all_articles.keys()

dict_keys(['status', 'totalResults', 'articles'])

In [8]:
len(all_articles['articles']), all_articles['articles']

(96,
 [{'source': {'id': None, 'name': 'Gizmodo.com'},
   'author': 'Matthew Phelan',
   'title': 'Some of the Most Hyped Gym Supplements Are Fake, Unregulated, or Both',
   'description': "Worse, a new survey of 375 sports medicine experts has uncovered that some of most popular and sketchiest products aren't even on their radar yet.",
   'url': 'https://gizmodo.com/some-of-the-most-hyped-gym-supplements-are-fake-unregulated-or-both-2000739118',
   'urlToImage': 'https://gizmodo.com/app/uploads/2026/03/multivitamin-1200x675.jpg',
   'publishedAt': '2026-03-27T19:05:43Z',
   'content': 'Doctors are only now starting to catch up on the weird, rapidly growing, and wildly unregulated world of underground bodybuilding supplements. Even with surging media attention on the rise of dubious… [+4820 chars]'},
  {'source': {'id': None, 'name': 'Gizmodo.com'},
   'author': 'Ed Cara',
   'title': 'Should You Be Taking Peptides?',
   'description': "From GLP-1s to anti-aging cocktails, peptides are

In [9]:
df = pd.DataFrame(all_articles['articles'])
df

,source,author,title,description,url,urlToImage,publishedAt,content
0,"{'id': None, 'name': 'Gizmodo.com'}",Matthew Phelan,"Some of the Most Hyped Gym Supplements Are Fake, Unregulated, or Both","Worse, a new survey of 375 sports medicine experts has uncovered that some of most popular and sketchiest products aren't even on their radar yet.",https://gizmodo.com/some-of-the-most-hyped-gym-supplements-are-fake-unregulated-or-both-2000739118,https://gizmodo.com/app/uploads/2026/03/multivitamin-1200x675.jpg,2026-03-27T19:05:43Z,"Doctors are only now starting to catch up on the weird, rapidly growing, and wildly unregulated world of underground bodybuilding supplements. Even with surging media attention on the rise of dubious… [+4820 chars]"
1,"{'id': None, 'name': 'Gizmodo.com'}",Ed Cara,Should You Be Taking Peptides?,"From GLP-1s to anti-aging cocktails, peptides are being widely marketed as wellness boosters. Here's what experts say you should know before you buy in.",https://gizmodo.com/should-you-be-taking-peptides-2000732422,https://gizmodo.com/app/uploads/2026/03/peptides-pipette-dropper-Giz-Asks-1200x675.jpg,2026-03-12T14:00:01Z,“Peptides” has fast become one of the biggest buzzwords in the wellness world. And they’re likely to get even more public attention in the coming months and years.\r\nPeptides are simple chains of two … [+11438 chars]
2,"{'id': None, 'name': 'Gizmodo.com'}",Ed Cara,"In Medical First, a Single Therapy Knocked Out 3 Autoimmune Diseases at Once","CAR T-cell therapy could help tackle the most severe cases that haven't responded to other treatments, her doctors say. \n\nALT DEK: 14 months later, the woman still doesn't need any treatments for her conditions.",https://gizmodo.com/in-medical-first-a-single-therapy-knocked-out-3-autoimmune-diseases-at-once-2000744222,https://gizmodo.com/app/uploads/2026/04/B-cell-antibodies-1200x675.jpg,2026-04-09T15:00:25Z,"In recent years, CAR T-cell therapy has become a revolutionary, if risky, treatment for some late-stage cancers. Yet CAR T may also have another life as a near-miraculous intervention for certain imm… [+4476 chars]"
3,"{'id': 'the-verge', 'name': 'The Verge'}",Victoria Song,How the Apple Watch defined modern health tech,"This is Optimizer, a weekly newsletter sent every Friday from Verge senior reviewer Victoria Song that dissects and discusses the latest gizmos and potions that swear they're going to change your life. Opt in for Optimizer here. You can trace the state of hea…",https://www.theverge.com/column/906391/apple-watch-optimizer-apple-50-health-tech-wearables,https://platform.theverge.com/wp-content/uploads/sites/2/2026/04/268248_APPLE_50_APPLE_WATCH_CVirginia.jpg?quality=90&strip=all&crop=0%2C10.732984293194%2C100%2C78.534031413613&w=1200,2026-04-03T12:52:14Z,"<ul><li></li><li></li><li></li></ul>\r\nDigital health screeners werent a thing until the Apple Watch. Its shaped how we think about wearables ever since.\r\nby\r\nVictoria SongClose\r\nSenior Reviewer, Wear… [+12217 chars]"
4,"{'id': 'business-insider', 'name': 'Business Insider'}",Jack Newsham,"Medvi, the AI-powered telehealth company, is fueled by ads from doctors who don't appear to exist",Medvi's marketing has included seemingly fake doctors and prompted lawsuits.,https://www.businessinsider.com/medvi-ai-weight-loss-millions-ai-advertising-legal-compliance-challenges-2026-4,https://i.insider.com/69d0328ae762ed6cfe44a3fb?width=1200&format=jpeg,2026-04-06T23:14:50Z,RICCARDO MILANI/Hans Lucas/AFP via Getty Images\r\n<ul><li>The New York Times reported that telehealth startup Medvi has over $1 billion in projected sales.</li><li>Founder Matthew Gallagher used AI to… [+7064 chars]
...,...,...,...,...,...,...,...,...
91,"{'id': None, 'name': 'Parade'}",Alexa Mellardo,Cardiologists Reveal How One Morning Habit Harms Your Heart Health,We're ALL guilty of this one.,https://parade.com/health/morning-habit-could-be-aging-your-heart-according-to-cardiologists,https://s.yimg.com/ny/api/res/1.2/4

In [10]:
df = df.rename(columns={
    "source": "Source Meta Data",
    "author": "Source",
    "publishedAt": "Date",
    "url": "URL",
    "urlToImage": "Image URL"
})

df["Query Domain"] = QUERY_CONFIGS[list_num]["query_domain"]

df

,Source Meta Data,Source,title,description,URL,Image URL,Date,content,Query Domain
0,"{'id': None, 'name': 'Gizmodo.com'}",Matthew Phelan,"Some of the Most Hyped Gym Supplements Are Fake, Unregulated, or Both","Worse, a new survey of 375 sports medicine experts has uncovered that some of most popular and sketchiest products aren't even on their radar yet.",https://gizmodo.com/some-of-the-most-hyped-gym-supplements-are-fake-unregulated-or-both-2000739118,https://gizmodo.com/app/uploads/2026/03/multivitamin-1200x675.jpg,2026-03-27T19:05:43Z,"Doctors are only now starting to catch up on the weird, rapidly growing, and wildly unregulated world of underground bodybuilding supplements. Even with surging media attention on the rise of dubious… [+4820 chars]",health
1,"{'id': None, 'name': 'Gizmodo.com'}",Ed Cara,Should You Be Taking Peptides?,"From GLP-1s to anti-aging cocktails, peptides are being widely marketed as wellness boosters. Here's what experts say you should know before you buy in.",https://gizmodo.com/should-you-be-taking-peptides-2000732422,https://gizmodo.com/app/uploads/2026/03/peptides-pipette-dropper-Giz-Asks-1200x675.jpg,2026-03-12T14:00:01Z,“Peptides” has fast become one of the biggest buzzwords in the wellness world. And they’re likely to get even more public attention in the coming months and years.\r\nPeptides are simple chains of two … [+11438 chars],health
2,"{'id': None, 'name': 'Gizmodo.com'}",Ed Cara,"In Medical First, a Single Therapy Knocked Out 3 Autoimmune Diseases at Once","CAR T-cell therapy could help tackle the most severe cases that haven't responded to other treatments, her doctors say. \n\nALT DEK: 14 months later, the woman still doesn't need any treatments for her conditions.",https://gizmodo.com/in-medical-first-a-single-therapy-knocked-out-3-autoimmune-diseases-at-once-2000744222,https://gizmodo.com/app/uploads/2026/04/B-cell-antibodies-1200x675.jpg,2026-04-09T15:00:25Z,"In recent years, CAR T-cell therapy has become a revolutionary, if risky, treatment for some late-stage cancers. Yet CAR T may also have another life as a near-miraculous intervention for certain imm… [+4476 chars]",health
3,"{'id': 'the-verge', 'name': 'The Verge'}",Victoria Song,How the Apple Watch defined modern health tech,"This is Optimizer, a weekly newsletter sent every Friday from Verge senior reviewer Victoria Song that dissects and discusses the latest gizmos and potions that swear they're going to change your life. Opt in for Optimizer here. You can trace the state of hea…",https://www.theverge.com/column/906391/apple-watch-optimizer-apple-50-health-tech-wearables,https://platform.theverge.com/wp-content/uploads/sites/2/2026/04/268248_APPLE_50_APPLE_WATCH_CVirginia.jpg?quality=90&strip=all&crop=0%2C10.732984293194%2C100%2C78.534031413613&w=1200,2026-04-03T12:52:14Z,"<ul><li></li><li></li><li></li></ul>\r\nDigital health screeners werent a thing until the Apple Watch. Its shaped how we think about wearables ever since.\r\nby\r\nVictoria SongClose\r\nSenior Reviewer, Wear… [+12217 chars]",health
4,"{'id': 'business-insider', 'name': 'Business Insider'}",Jack Newsham,"Medvi, the AI-powered telehealth company, is fueled by ads from doctors who don't appear to exist",Medvi's marketing has included seemingly fake doctors and prompted lawsuits.,https://www.businessinsider.com/medvi-ai-weight-loss-millions-ai-advertising-legal-compliance-challenges-2026-4,https://i.insider.com/69d0328ae762ed6cfe44a3fb?width=1200&format=jpeg,2026-04-06T23:14:50Z,RICCARDO MILANI/Hans Lucas/AFP via Getty Images\r\n<ul><li>The New York Times reported that telehealth startup Medvi has over $1 billion in projected sales.</li><li>Founder Matthew Gallagher used AI to… [+7064 chars],health
...,...,...,...,...,...,...,...,...,...
91,"{'id': None, 'name': 'Parade'}",Alexa Mellardo,Cardiologists Reveal How One Morning Habit Harms Your Heart Health,We're ALL guilty of this one.,https://parade.com/health/morning-habit-could-be-aging-your-heart-accordin

In [11]:
TRUNC_PATTERN = re.compile(r"(\u2026|\.\.\.)\s*\[\+\d+\s+chars\]")

def clean_base_sentence(text: str) -> str:
    if not isinstance(text, str):
        return text

    # ✅ REMOVE NewsAPI truncation marker (ONLY in cleaned version)
    text = TRUNC_PATTERN.sub("", text)

    # Decode HTML entities
    text = html.unescape(text)

    # Remove UL/LI tags
    text = re.sub(r"</?ul>", "", text)
    text = re.sub(r"</?li>", " ", text)

    # Normalize whitespace
    text = re.sub(r"[\r\n]+", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [12]:
def build_prediction_dataframe(df):
    df = df.reset_index(drop=True).copy()
    df["article_id"] = df.index

    prediction_keywords = [
        "expected", "forecast", "forecasted", "forecasting",
        "forecast estimate", "forecasted outcome", "projected",
        "projection", "predict", "predicted", "estimate",
        "expectation", "anticipation", "prognosis",
        "speculation", "foretelling", "prophecy", "guess"
    ]

    pattern = r"\b(" + "|".join(map(re.escape, prediction_keywords)) + r")\b"

    df["prediction_visible"] = (
        (
            df["title"].fillna("") +
            df["description"].fillna("") +
            df["content"].fillna("")
        )
        .str.lower()
        .str.contains(pattern, regex=True)
    ).astype(int)

    parts = []
    for field, order in [("title", 0), ("description", 1), ("content", 2)]:
        tmp = df.copy()
        tmp["Base Sentence (raw)"] = tmp[field]   # ✅ RAW PRESERVED
        tmp["Source_Field"] = field
        tmp["field_order"] = order
        parts.append(tmp)

    long_df = pd.concat(parts, ignore_index=True)

    # ✅ CLEANED COLUMN (used downstream)
    long_df["Base Sentence"] = long_df["Base Sentence (raw)"].apply(clean_base_sentence)

    # ✅ Sentence splitting uses CLEAN text
    sfe = SpacyFeatureExtraction(long_df, "Base Sentence")
    long_df = sfe.split_into_sentences()

    # ✅ Label per sentence
    long_df["Sentence Label"] = (
        long_df["Base Sentence"]
        .str.lower()
        .str.contains(pattern, regex=True)
        .astype(int)
    )

    long_df["Human Annotation"] = ""
    long_df["Human Reasoning"] = ""

    priority_cols = [
        "Base Sentence",
        "Base Sentence (raw)",
        "Sentence Label",
        "Human Annotation",
        "Human Reasoning",
        "Source",
        "Date"
    ]

    priority_cols = [c for c in priority_cols if c in long_df.columns]
    remaining_cols = [c for c in long_df.columns if c not in priority_cols]

    return long_df[priority_cols + remaining_cols]

In [13]:
final_df = build_prediction_dataframe(df)
final_df.head(3)

/var/folders/78/9z0b45fx1xqbwxh8vk97lcfh0000gn/T/ipykernel_43161/543514960.py:22: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(pattern, regex=True)
/var/folders/78/9z0b45fx1xqbwxh8vk97lcfh0000gn/T/ipykernel_43161/543514960.py:46: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(pattern, regex=True)


,Base Sentence,Base Sentence (raw),Sentence Label,Human Annotation,Human Reasoning,Source,Date,Source Meta Data,title,description,URL,Image URL,content,Query Domain,article_id,prediction_visible,Source_Field,field_order
0,"Some of the Most Hyped Gym Supplements Are Fake, Unregulated, or Both","Some of the Most Hyped Gym Supplements Are Fake, Unregulated, or Both",0,,,Matthew Phelan,2026-03-27T19:05:43Z,"{'id': None, 'name': 'Gizmodo.com'}","Some of the Most Hyped Gym Supplements Are Fake, Unregulated, or Both","Worse, a new survey of 375 sports medicine experts has uncovered that some of most popular and sketchiest products aren't even on their radar yet.",https://gizmodo.com/some-of-the-most-hyped-gym-supplements-are-fake-unregulated-or-both-2000739118,https://gizmodo.com/app/uploads/2026/03/multivitamin-1200x675.jpg,"Doctors are only now starting to catch up on the weird, rapidly growing, and wildly unregulated world of underground bodybuilding supplements. Even with surging media attention on the rise of dubious… [+4820 chars]",health,0,0,title,0
1,Should You Be Taking Peptides?,Should You Be Taking Peptides?,0,,,Ed Cara,2026-03-12T14:00:01Z,"{'id': None, 'name': 'Gizmodo.com'}",Should You Be Taking Peptides?,"From GLP-1s to anti-aging cocktails, peptides are being widely marketed as wellness boosters. Here's what experts say you should know before you buy in.",https://gizmodo.com/should-you-be-taking-peptides-2000732422,https://gizmodo.com/app/uploads/2026/03/peptides-pipette-dropper-Giz-Asks-1200x675.jpg,“Peptides” has fast become one of the biggest buzzwords in the wellness world. And they’re likely to get even more public attention in the coming months and years.\r\nPeptides are simple chains of two … [+11438 chars],health,1,0,title,0
2,"In Medical First, a Single Therapy Knocked Out 3 Autoimmune Diseases at Once","In Medical First, a Single Therapy Knocked Out 3 Autoimmune Diseases at Once",0,,,Ed Cara,2026-04-09T15:00:25Z,"{'id': None, 'name': 'Gizmodo.com'}","In Medical First, a Single Therapy Knocked Out 3 Autoimmune Diseases at Once","CAR T-cell therapy could help tackle the most severe cases that haven't responded to other treatments, her doctors say. \n\nALT DEK: 14 months later, the woman still doesn't need any treatments for her conditions.",https://gizmodo.com/in-medical-first-a-single-therapy-knocked-out-3-autoimmune-diseases-at-once-2000744222,https://gizmodo.com/app/uploads/2026/04/B-cell-antibodies-1200x675.jpg,"In recent years, CAR T-cell therapy has become a revolutionary, if risky, treatment for some late-stage cancers. Yet CAR T may also have another life as a near-miraculous intervention for certain imm… [+4476 chars]",health,2,0,title,0


In [14]:
predictions_df = final_df[final_df["Sentence Label"] == 1]
predictions_df.head(3)

,Base Sentence,Base Sentence (raw),Sentence Label,Human Annotation,Human Reasoning,Source,Date,Source Meta Data,title,description,URL,Image URL,content,Query Domain,article_id,prediction_visible,Source_Field,field_order
263,RICCARDO MILANI/Hans Lucas/AFP via Getty Images The New York Times reported that telehealth startup Medvi has over $1 billion in projected sales.,RICCARDO MILANI/Hans Lucas/AFP via Getty Images\r\n<ul><li>The New York Times reported that telehealth startup Medvi has over $1 billion in projected sales.</li><li>Founder Matthew Gallagher used AI to… [+7064 chars],1,,,Jack Newsham,2026-04-06T23:14:50Z,"{'id': 'business-insider', 'name': 'Business Insider'}","Medvi, the AI-powered telehealth company, is fueled by ads from doctors who don't appear to exist",Medvi's marketing has included seemingly fake doctors and prompted lawsuits.,https://www.businessinsider.com/medvi-ai-weight-loss-millions-ai-advertising-legal-compliance-challenges-2026-4,https://i.insider.com/69d0328ae762ed6cfe44a3fb?width=1200&format=jpeg,RICCARDO MILANI/Hans Lucas/AFP via Getty Images\r\n<ul><li>The New York Times reported that telehealth startup Medvi has over $1 billion in projected sales.</li><li>Founder Matthew Gallagher used AI to… [+7064 chars],health,4,1,content,2
378,"Online sports betting is more popular than ever, with Americans expected to legally wager billions of dollars on this year's March Madness basketball tournament.","Online sports betting is more popular than ever, with Americans expected to legally wager billions of dollars on this year's March Madness basketball tournament. But a growing body of evidence reveal… [+5473 chars]",1,,,Alana Wise,2026-04-04T09:15:00Z,"{'id': None, 'name': 'NPR'}","When legal sports betting surges, so do Americans' financial problems","As online betting has grown in popularity, a new report from the New York Federal Reserve builds on the troubling link between legal sports wagering and financial health.",https://www.npr.org/2026/04/04/nx-s1-5773354/legal-sports-betting-research-credit-bankruptcy,https://npr.brightspotcdn.com/dims3/default/strip/false/crop/5037x2833+0+262/resize/1400/quality/85/format/jpeg/?url=http%3A%2F%2Fnpr-brightspot.s3.amazonaws.com%2F1d%2F75%2F55a395ee403e89e2c4c282794138%2Fap26022534309429.jpg,"Online sports betting is more popular than ever, with Americans expected to legally wager billions of dollars on this year's March Madness basketball tournament. But a growing body of evidence reveal… [+5473 chars]",health,58,1,content,2


In [15]:
predictions_df["Base Sentence"] = predictions_df["Base Sentence"].apply(clean_base_sentence)
predictions_df.head(3)

,Base Sentence,Base Sentence (raw),Sentence Label,Human Annotation,Human Reasoning,Source,Date,Source Meta Data,title,description,URL,Image URL,content,Query Domain,article_id,prediction_visible,Source_Field,field_order
263,RICCARDO MILANI/Hans Lucas/AFP via Getty Images The New York Times reported that telehealth startup Medvi has over $1 billion in projected sales.,RICCARDO MILANI/Hans Lucas/AFP via Getty Images\r\n<ul><li>The New York Times reported that telehealth startup Medvi has over $1 billion in projected sales.</li><li>Founder Matthew Gallagher used AI to… [+7064 chars],1,,,Jack Newsham,2026-04-06T23:14:50Z,"{'id': 'business-insider', 'name': 'Business Insider'}","Medvi, the AI-powered telehealth company, is fueled by ads from doctors who don't appear to exist",Medvi's marketing has included seemingly fake doctors and prompted lawsuits.,https://www.businessinsider.com/medvi-ai-weight-loss-millions-ai-advertising-legal-compliance-challenges-2026-4,https://i.insider.com/69d0328ae762ed6cfe44a3fb?width=1200&format=jpeg,RICCARDO MILANI/Hans Lucas/AFP via Getty Images\r\n<ul><li>The New York Times reported that telehealth startup Medvi has over $1 billion in projected sales.</li><li>Founder Matthew Gallagher used AI to… [+7064 chars],health,4,1,content,2
378,"Online sports betting is more popular than ever, with Americans expected to legally wager billions of dollars on this year's March Madness basketball tournament.","Online sports betting is more popular than ever, with Americans expected to legally wager billions of dollars on this year's March Madness basketball tournament. But a growing body of evidence reveal… [+5473 chars]",1,,,Alana Wise,2026-04-04T09:15:00Z,"{'id': None, 'name': 'NPR'}","When legal sports betting surges, so do Americans' financial problems","As online betting has grown in popularity, a new report from the New York Federal Reserve builds on the troubling link between legal sports wagering and financial health.",https://www.npr.org/2026/04/04/nx-s1-5773354/legal-sports-betting-research-credit-bankruptcy,https://npr.brightspotcdn.com/dims3/default/strip/false/crop/5037x2833+0+262/resize/1400/quality/85/format/jpeg/?url=http%3A%2F%2Fnpr-brightspot.s3.amazonaws.com%2F1d%2F75%2F55a395ee403e89e2c4c282794138%2Fap26022534309429.jpg,"Online sports betting is more popular than ever, with Americans expected to legally wager billions of dollars on this year's March Madness basketball tournament. But a growing body of evidence reveal… [+5473 chars]",health,58,1,content,2


In [16]:
predictions_df

,Base Sentence,Base Sentence (raw),Sentence Label,Human Annotation,Human Reasoning,Source,Date,Source Meta Data,title,description,URL,Image URL,content,Query Domain,article_id,prediction_visible,Source_Field,field_order
263,RICCARDO MILANI/Hans Lucas/AFP via Getty Images The New York Times reported that telehealth startup Medvi has over $1 billion in projected sales.,RICCARDO MILANI/Hans Lucas/AFP via Getty Images\r\n<ul><li>The New York Times reported that telehealth startup Medvi has over $1 billion in projected sales.</li><li>Founder Matthew Gallagher used AI to… [+7064 chars],1,,,Jack Newsham,2026-04-06T23:14:50Z,"{'id': 'business-insider', 'name': 'Business Insider'}","Medvi, the AI-powered telehealth company, is fueled by ads from doctors who don't appear to exist",Medvi's marketing has included seemingly fake doctors and prompted lawsuits.,https://www.businessinsider.com/medvi-ai-weight-loss-millions-ai-advertising-legal-compliance-challenges-2026-4,https://i.insider.com/69d0328ae762ed6cfe44a3fb?width=1200&format=jpeg,RICCARDO MILANI/Hans Lucas/AFP via Getty Images\r\n<ul><li>The New York Times reported that telehealth startup Medvi has over $1 billion in projected sales.</li><li>Founder Matthew Gallagher used AI to… [+7064 chars],health,4,1,content,2
378,"Online sports betting is more popular than ever, with Americans expected to legally wager billions of dollars on this year's March Madness basketball tournament.","Online sports betting is more popular than ever, with Americans expected to legally wager billions of dollars on this year's March Madness basketball tournament. But a growing body of evidence reveal… [+5473 chars]",1,,,Alana Wise,2026-04-04T09:15:00Z,"{'id': None, 'name': 'NPR'}","When legal sports betting surges, so do Americans' financial problems","As online betting has grown in popularity, a new report from the New York Federal Reserve builds on the troubling link between legal sports wagering and financial health.",https://www.npr.org/2026/04/04/nx-s1-5773354/legal-sports-betting-research-credit-bankruptcy,https://npr.brightspotcdn.com/dims3/default/strip/false/crop/5037x2833+0+262/resize/1400/quality/85/format/jpeg/?url=http%3A%2F%2Fnpr-brightspot.s3.amazonaws.com%2F1d%2F75%2F55a395ee403e89e2c4c282794138%2Fap26022534309429.jpg,"Online sports betting is more popular than ever, with Americans expected to legally wager billions of dollars on this year's March Madness basketball tournament. But a growing body of evidence reveal… [+5473 chars]",health,58,1,content,2


In [17]:
len(predictions_df)

2

In [ ]:
def build_query_package(
    *,
    query_string: str,
    query_domain: str, 
    start_date: str,
    end_date: str,
    df,
    final_df,
    predictions_df,
    language: str = "en",
    sort_by: str = "relevancy",
    page_size: int = None
):
    """
    Builds a query-aligned package and a deterministic filename prefix.
    """

    # -------------------------
    # Prefix from query
    # -------------------------
    base_prefix = (
        query_string.lower()
        .replace(" or ", "_")
        .replace(" and ", "_")
        .replace(" ", "_")
    )
    base_prefix = re.sub(r"[^a-z0-9_]", "", base_prefix)
    base_prefix = re.sub(r"_+", "_", base_prefix).strip("_")

    prefix = f"{base_prefix}_{start_date}_to_{end_date}"

    # -------------------------
    # Query metadata
    # -------------------------
    query_meta = {
        "query": query_string,
        "query_domain": query_domain,
        "language": language,
        "from_date": start_date,
        "to_date": end_date,
        "sort_by": sort_by,
        "page_size": page_size,
        "timestamp_utc": datetime.utcnow().isoformat()
    }

    # -------------------------
    # Full package
    # -------------------------
    package = {
        "query_meta": query_meta,
        "counts": {
            "num_raw_articles": len(df),
            "num_sentences": len(final_df),
            "num_potential_predictions": len(predictions_df)
        },
        "raw_articles": df.to_dict(orient="records"),
        "processed_sentences": final_df.to_dict(orient="records"),
        "potential_predictions": predictions_df.to_dict(orient="records")
    }

    return prefix, package

In [ ]:
if len(predictions_df) >= 1: 
    prefix, query_package = build_query_package(
        query_string=QUERY_CONFIGS[list_num]["query"],
        query_domain=QUERY_CONFIGS[list_num]["query_domain"],
        start_date=start_date,
        end_date=end_date,
        df=df,
        final_df=final_df,
        predictions_df=predictions_df,
        page_size=7
    )

In [ ]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
save_path = os.path.join(base_data_path, 'news_api', 'annotators')

if len(predictions_df) >= 1: 
    DataProcessing.save_to_file(
        data=predictions_df,
        path=save_path,
        prefix=f"{prefix}_predictions",
        save_file_type="csv"
    )

In [ ]:
if len(predictions_df) >= 1: 
    DataProcessing.save_to_file(
        data=query_package,
        path=save_path,
        prefix=f"{prefix}_full",
        save_file_type="json"
    )